In [1]:
!pip install easyocr

In [2]:
!pip install torch --index-url https://download.pytorch.org/whl/cu129

Looking in indexes: https://download.pytorch.org/whl/cu129


In [3]:
!git clone https://github.com/daoanhkhoa123/OCR_Vinmec.git
%cd /kaggle/working/OCR_Vinmec

Cloning into 'OCR_Vinmec'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 53 (delta 19), reused 15 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 11.27 MiB | 15.83 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/kaggle/working/OCR_Vinmec


In [9]:
import argparse
import gc
import os
import time

import cv2
import torch

from engine.ultils import align_by_hough
from engine.easyocr_reader import crop_by_keywords
from engine.vlm_reader import get_first_info, get_second_info, load_vlm_model
from engine.checkboxes import detect_checkboxes, classify_checkboxes, summarize_checkbox


def run_pipeline(image_path, aligned_image_path):
    pipeline_start = time.perf_counter()

    # Step 1: Align Image
    t0 = time.perf_counter()
    rotated, angle = align_by_hough(cv2.imread(image_path))
    print(f"Rotation angle: {angle} | Time: {time.perf_counter() - t0:.3f}s")

    cv2.imwrite(aligned_image_path, rotated)
    print(f"Saved to {aligned_image_path}")

    # Step 2: Crop by Keywords
    t0 = time.perf_counter()
    img = cv2.imread(aligned_image_path)
    cropped = crop_by_keywords(img)
    print(f"Crop by keywords | Time: {time.perf_counter() - t0:.3f}s")

    # Step 3: VLM Extraction
    t0 = time.perf_counter()
    first_info = get_first_info(cropped)
    print(first_info)
    print(f"VLM First Info | Time: {time.perf_counter() - t0:.3f}s")

    t0 = time.perf_counter()
    second_info = get_second_info(cropped)
    print(second_info)
    print(f"VLM Second Info | Time: {time.perf_counter() - t0:.3f}s")

    # Step 4: Checkbox Detection & Classification
    t0 = time.perf_counter()
    stats, labels = detect_checkboxes(cropped)
    checkbox_dict = classify_checkboxes(cropped, stats, n_groups=13)
    checkbox_info = summarize_checkbox(checkbox_dict)
    print(checkbox_info)
    print(f"Checkbox Processing | Time: {time.perf_counter() - t0:.3f}s")

    # Final Summary
    info = {**first_info, **second_info, **checkbox_info}
    print("\nMerged Output:", info)

    pipeline_duration = time.perf_counter() - pipeline_start
    print(f"\n==========================================")
    print(f"Total Pipeline Execution Time: {pipeline_duration:.3f}s")
    print(f"==========================================")

    return info



In [13]:
info  = run_pipeline("/kaggle/working/OCR_Vinmec/handwriting_test.png", "/kaggle/working/OCR_Vinmec/handwriting_test_aligned.png")
print(info)

Rotation angle: 0.9999942779541016 | Time: 0.376s
Saved to /kaggle/working/OCR_Vinmec/handwriting_test_aligned.png
Crop by keywords | Time: 4.473s
{'Họ và tên': 'VŨ ĐÌNH TÔN', 'Ngày/tháng/năm sinh': '02/10', 'Lớp': None}
VLM First Info | Time: 15.235s
{'Thị lực nhìn xa không kính': {'Mắt Phải': '/10', 'Mắt Trái': '/10'}, 'Thị lực nhìn xa có kính': {'Mắt Phải': '/10', 'Mắt Trái': '/10'}}
VLM Second Info | Time: 18.390s
{'Giới tính': 'Nam', 'Đang điều trị kiểm soát cận thị': None, 'KIỂM TRA KÍNH ĐANG ĐEO': 'Gọng cong vênh', 'Đang đeo kính': 'Có', 'Đánh giá thị lực nhìn xa': 'Đạt', 'Thị lực nhìn gần (áp dụng với trẻ từ 4 đến 9 tuổi)': 'Không đạt', 'Kết quả thị lực': 'Đạt', 'Kiểm tra sắc giác': 'Đạt', 'Hai mắt chính thị hoặc ít nguy cơ': 'Hai mắt chính thị hoặc ít nguy cơ', 'Nghi ngờ tật khúc xạ/bệnh lý mắt, cần khám để chẩn đoán xác định': None, 'Có tật khúc xạ, kính đang đeo phù hợp': 'Có tật khúc xạ, kính đang đeo phù hợp', 'Có tật khúc xạ, kính chưa tối ưu, cần kiểm tra lại kính đang đ

In [14]:
import unicodedata


def _flatten(d, parent_key=""):
    items = {}
    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(_flatten(v, key))
        else:
            items[key] = v
    return items


def _normalize(value):
    if value is None:
        return ""
    return unicodedata.normalize("NFC", str(value)).strip()


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    previous_row = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current_row = [i] + [0] * len(b)
        for j, cb in enumerate(b, start=1):
            current_row[j] = min(
                current_row[j - 1] + 1,            # insertion
                previous_row[j] + 1,                # deletion
                previous_row[j - 1] + (ca != cb),   # substitution
            )
        previous_row = current_row
    return previous_row[-1]


def evaluate(gtruth: dict, pred: dict):
    gt_flat = _flatten(gtruth)
    pred_flat = _flatten(pred)
    all_keys = sorted(set(gt_flat) | set(pred_flat))

    per_field = {}
    correct = 0
    total_edit_distance = 0
    total_ref_chars = 0

    for key in all_keys:
        gt_value = _normalize(gt_flat.get(key))
        pred_value = _normalize(pred_flat.get(key))

        is_correct = gt_value == pred_value
        correct += int(is_correct)

        dist = levenshtein(pred_value, gt_value)
        ref_len = len(gt_value)
        field_cer = dist / ref_len if ref_len else float(bool(dist))

        total_edit_distance += dist
        total_ref_chars += ref_len

        per_field[key] = {
            "gtruth": gt_value,
            "pred": pred_value,
            "correct": is_correct,
            "edit_distance": dist,
            "cer": field_cer,
        }

    num_fields = len(all_keys)
    return {
        "field_accuracy": correct / num_fields if num_fields else 1.0,
        "corpus_cer": total_edit_distance / total_ref_chars if total_ref_chars else 0.0,
        "num_fields": num_fields,
        "num_correct": correct,
        "per_field": per_field,
    }


gtruth = {"Họ và tên": "VŨ ĐÌNH TOÀN", "Ngày/tháng/năm sinh": "02/09/1990", "Lớp": "MiRA 1", "Thị lực nhìn xa không kính": {"Mắt Phải": "8/10", "Mắt Trái": "7/10"}, "Thị lực nhìn xa có kính": {"Mắt Phải": "8/10", "Mắt Trái": "2/10"}, "Giới tính": "Nam", "Đang điều trị kiểm soát cận thị": "Không", "KIỂM TRA KÍNH ĐANG ĐEO": "Gọng cong vênh", "Đang đeo kính": "Có", "Đánh giá thị lực nhìn xa": "Đạt", "Thị lực nhìn gần (áp dụng với trẻ từ 4 đến 9 tuổi)": "Không đạt", "Kết quả thị lực": "Đạt", "Kiểm tra sắc giác": "Đạt", "Hai mắt chính thị hoặc ít nguy cơ": "Hai mắt chính thị hoặc ít nguy cơ", "Nghi ngờ tật khúc xạ/bệnh lý mắt, cần khám để chẩn đoán xác định": None, "Có tật khúc xạ, kính đang đeo phù hợp": "Có tật khúc xạ, kính đang đeo phù hợp", "Có tật khúc xạ, kính chưa tối ưu, cần kiểm tra lại kính đang đeo": None, "Kết luận khám": "Ít nguy cơ (A và C)"}
pred = info
result = evaluate(gtruth, pred)
print(f"Field accuracy: {result['num_correct']}/{result['num_fields']} = {result['field_accuracy']:.2%}")
print(f"Corpus-level CER: {result['corpus_cer']:.2%}\n")

for key, info in result["per_field"].items():
    status = "OK  " if info["correct"] else "MISS"
    print(f"[{status}] {key}: gt={info['gtruth']!r} pred={info['pred']!r} cer={info['cer']:.2%}")


Field accuracy: 12/20 = 60.00%
Corpus-level CER: 12.57%

[OK  ] Có tật khúc xạ, kính chưa tối ưu, cần kiểm tra lại kính đang đeo: gt='' pred='' cer=0.00%
[OK  ] Có tật khúc xạ, kính đang đeo phù hợp: gt='Có tật khúc xạ, kính đang đeo phù hợp' pred='Có tật khúc xạ, kính đang đeo phù hợp' cer=0.00%
[OK  ] Giới tính: gt='Nam' pred='Nam' cer=0.00%
[OK  ] Hai mắt chính thị hoặc ít nguy cơ: gt='Hai mắt chính thị hoặc ít nguy cơ' pred='Hai mắt chính thị hoặc ít nguy cơ' cer=0.00%
[MISS] Họ và tên: gt='VŨ ĐÌNH TOÀN' pred='VŨ ĐÌNH TÔN' cer=16.67%
[OK  ] KIỂM TRA KÍNH ĐANG ĐEO: gt='Gọng cong vênh' pred='Gọng cong vênh' cer=0.00%
[OK  ] Kiểm tra sắc giác: gt='Đạt' pred='Đạt' cer=0.00%
[OK  ] Kết luận khám: gt='Ít nguy cơ (A và C)' pred='Ít nguy cơ (A và C)' cer=0.00%
[OK  ] Kết quả thị lực: gt='Đạt' pred='Đạt' cer=0.00%
[MISS] Lớp: gt='MiRA 1' pred='' cer=100.00%
[OK  ] Nghi ngờ tật khúc xạ/bệnh lý mắt, cần khám để chẩn đoán xác định: gt='' pred='' cer=0.00%
[MISS] Ngày/tháng/năm sinh: gt='02/09/